# Attribution graph with transcoders on GPT-2 small

A transcoder is a sparse, wide stand-in for one MLP: it reads the MLP's input and writes
the MLP's output through a few active features. Swap every MLP for its transcoder, hold
the attention patterns and the normalisation scales at their captured values, and the
forward pass becomes linear in the features; the direct effect of every feature, token
embedding and leftover error on every later feature and on the output logits is then an
exact edge. That graph is an attribution graph.

Papers:

- Ameisen, Lindsey, Pearce, Gurnee, Turner, Chen, Citro, Abrahams, Carter, Hosmer, Marcus,
  Sklar, Templeton, Bricken, McDougall, Cunningham, Henighan, Jermyn, Jones, Persic, Qi,
  Thompson, Zimmerman, Rivoire, Conerly, Olah and Batson (2025), "Circuit Tracing:
  Revealing Computational Graphs in Language Models", Transformer Circuits Thread,
  https://transformer-circuits.pub/2025/attribution-graphs/methods.html
- Dunefsky, Chlenski and Nanda (2024), "Transcoders Find Interpretable LLM Feature
  Circuits", NeurIPS 2024, https://arxiv.org/abs/2406.11944

The transcoders are the GPT-2 small set Dunefsky et al. released (24,576 features per
layer, all 12 layers), in the first author's safetensors repackaging
`jacobdunefsky/gpt2small-transcoders` on the Hugging Face Hub (original release
`pchlenski/gpt2-transcoders`). They were trained on TransformerLens's GPT-2 small with its
default weight processing, so the model here is loaded the same way. Expect a download of
about 0.5 GB for GPT-2 and 1.7 GB for the transcoders on first run; after that the
notebook runs on a four-core laptop CPU in about five minutes (the graph itself takes three).

```text
pip install torchlens "transformer-lens==3.8.0" huggingface_hub safetensors
```

In [ ]:
# recipe-test: tool=transcoders expected=True
import torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from torch import nn
from transformer_lens import HookedTransformer

import torchlens as tl

torch.manual_seed(0)
# fold_ln, center_writing_weights and center_unembed: the setup the transcoders were trained on.
model = HookedTransformer.from_pretrained("gpt2", device="cpu").eval()

REPO = "jacobdunefsky/gpt2small-transcoders"
REVISION = "8f254a3a8cda641cb078137415b6b9503e10fc6b"


class Transcoder(nn.Module):
    """One GPT-2 small MLP transcoder, in the SAE Lens orientation (W_enc is d_in x features)."""

    def __init__(self, layer: int) -> None:
        super().__init__()
        name = f"gpt2-small-dun-chl-mlp-tc{layer}/sae.safetensors"
        weights = load_file(hf_hub_download(REPO, name, revision=REVISION))
        self.W_enc, self.b_enc = weights["W_enc"], weights["b_enc"]
        self.W_dec, self.b_dec = weights["W_dec"], weights["b_dec"]

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return torch.relu(x @ self.W_enc + self.b_enc)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.encode(x) @ self.W_dec + self.b_dec


transcoders = {f"blocks.{layer}.mlp": Transcoder(layer) for layer in range(12)}

prompt = "When Mary and John went to the store, John gave a drink to"
tokens = model.to_tokens(prompt)  # BOS first
mary = model.to_single_token(" Mary")
RESULT = int(model(tokens)[0, -1].argmax()) == mary

## 1. Splice the transcoders in: the replacement model

`tl.splice_module(transcoder, input_route="in")` on an MLP module runs the transcoder on
the arguments the MLP was called with and uses its result as the MLP's output. With all
twelve spliced in, the transcoders alone miss part of each MLP's computation, and on this
prompt that is enough to lose the answer. Adding back each MLP's error on this prompt (its
output minus the transcoder's) gives the local replacement model of Ameisen et al.: it
reproduces the clean run exactly, and the graph's error nodes are those same error terms.

In [ ]:
# recipe-test: tool=transcoders expected=True
class LocalReplacement(nn.Module):
    """A transcoder plus a fixed error term; optionally one feature zeroed at one position."""

    def __init__(self, base: Transcoder, error: torch.Tensor, feature=None, position=None) -> None:
        super().__init__()
        self.base, self.error, self.feature, self.position = base, error, feature, position

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        acts = self.base.encode(x)
        if self.feature is not None:
            acts = acts.clone()
            acts[..., self.position, self.feature] = 0
        return acts @ self.base.W_dec + self.base.b_dec + self.error


def run_with(modules: dict) -> torch.Tensor:
    """Last-position logits with each named MLP replaced by its module."""
    hooks = {tl.module(site): tl.splice_module(m, input_route="in") for site, m in modules.items()}
    spliced = tl.trace(
        model, tokens, capture=tl.options.CaptureOptions(replayable=True, hooks=hooks)
    )
    return spliced.output_ops[0].out[0, -1]


def kl_from_clean(logits: torch.Tensor) -> float:
    return float(
        (clean_logits.softmax(-1) * (clean_logits.log_softmax(-1) - logits.log_softmax(-1))).sum()
    )


clean = tl.trace(model, tokens)
clean_logits = clean.output_ops[0].out[0, -1]
errors = {}
with torch.no_grad():
    for site, tc in transcoders.items():
        mlp_in, mlp_out = clean[site.replace(".mlp", ".ln2")].out, clean[site].out
        errors[site] = mlp_out - tc(mlp_in)
        out, recon = (
            mlp_out[0, 1:],
            mlp_out[0, 1:] - errors[site][0, 1:],
        )  # skip BOS: its norm dwarfs the rest
        explained = 1 - (out - recon).pow(2).sum() / (out - out.mean(0)).pow(2).sum()
        active = (tc.encode(mlp_in[0, 1:]) > 0).sum(-1).float().mean()
        print(
            f"{site:14s} variance explained {explained:6.3f}   active features per token {active:6.1f}"
        )

alone = run_with(transcoders)
local = run_with({site: LocalReplacement(tc, errors[site]) for site, tc in transcoders.items()})
print(
    f"transcoders alone: top token {model.to_string(int(alone.argmax()))!r}, KL from clean {kl_from_clean(alone):.3f} nats"
)
print(
    f"local replacement: top token {model.to_string(int(local.argmax()))!r}, KL from clean {kl_from_clean(local):.1e} nats"
)
RESULT = int(local.argmax()) == mary and kl_from_clean(local) < 1e-3

## 2. Build the attribution graph

`tl.attribution.build_graph` reads the transcoders' features off the clean trace and computes
every edge in chunked backward passes along the frozen graph. `frozen=` names what is held
at its captured value: in TransformerLens's GPT-2 each attention pattern is the `softmax` op
in `blocks.{L}.attn`, and each normalisation scale is the `sqrt` op in the `ln1`, `ln2` and
`ln_final` modules (`ln1` runs three times per block, once each for the query, key and value
inputs). Iterating the trace finds their op labels. With `target=None` the graph
seeds the most probable next tokens.

In [ ]:
# recipe-test: tool=transcoders expected=True
frozen = []
for op in clean:
    module = str(op.module or "").split(":")[0]
    is_pattern = op.func_name == "softmax" and module.endswith(".attn")
    is_scale = op.func_name == "sqrt" and module.split(".")[-1] in {"ln1", "ln2", "ln_final"}
    if is_pattern or is_scale:
        frozen.append(str(op.label))
print(len(frozen), "frozen ops")

graph = tl.attribution.build_graph(
    clean,
    transcoders,
    embedding_module="embed",
    frozen=frozen,
    max_num_logits=3,
)
print(graph.node_counts)
for token_id, p in zip(graph.logit_token_ids.tolist(), graph.logit_probabilities.tolist()):
    print(f"  logit node {model.to_string(token_id)!r}: p={p:.3f}")
# 12 patterns; 49 scales: TransformerLens normalises the attention input three times (query, key, value).
RESULT = len(frozen) == 12 + 49 and mary in graph.logit_token_ids.tolist()

## 3. Read it: what writes the " Mary" logit

Row `t` of `adjacency_matrix` holds the direct effect of every node on node `t`. Every
feature and logit node is fully accounted for: its row plus the named constants equals its
value, so a graph whose `other` constant is small next to the logit has no path that
bypasses the nodes.

In [ ]:
# recipe-test: tool=transcoders expected=True
A = graph.adjacency_matrix
logit_rows = graph.node_indices("logit")
row = logit_rows[graph.logit_token_ids.tolist().index(mary)]


def describe(index: int) -> str:
    node = graph.nodes[index]
    where = f"pos {node.position} {model.to_string(tokens[0, node.position])!r}"
    if node.kind == "feature":
        return f"feature {node.feature_index:5d} of {node.site} at {where}"
    return f"{node.kind} {node.site} at {where}"


top = A[row].abs().topk(10).indices.tolist()
for index in top:
    print(f"{A[row, index]:+8.3f}  {describe(index)}")

accounted = A.sum(1) + sum(c.nan_to_num() for c in graph.constants.values())
seeded = graph.node_indices("feature") + logit_rows
gap = (accounted[seeded] - graph.node_values[seeded]).abs().max()
print(f"largest accounting gap over {len(seeded)} nodes: {gap:.2e}")
print(
    {name: round(float(c[row]), 3) for name, c in graph.constants.items()},
    "value",
    round(float(graph.node_values[row]), 3),
)
error_share = A[row, graph.node_indices("error")].abs().sum() / A[row].abs().sum()
print(f"share of the |edges| into ' Mary' carried by error nodes: {error_share:.2f}")
RESULT = bool(gap < 1e-2 * graph.node_values[seeded].abs().max())

## 4. Check one edge causally

The strongest feature edge into " Mary" is a claim: remove that feature and the logit should
drop. Zero the feature at its position inside the local replacement model (every later
transcoder recomputes its features; the error terms stay fixed) and compare the " Mary" logit
with the unedited local replacement.

In [ ]:
# recipe-test: tool=transcoders expected=True
feature_rows = set(graph.node_indices("feature"))
best = next(i for i in top if i in feature_rows and A[row, i] > 0)
node = graph.nodes[best]
edited = {site: LocalReplacement(tc, errors[site]) for site, tc in transcoders.items()}
edited[node.site] = LocalReplacement(
    transcoders[node.site], errors[node.site], feature=node.feature_index, position=node.position
)
drop = float(local[mary] - run_with(edited)[mary])
print(
    f"{describe(best)}: direct edge {A[row, best]:+.3f}, ablation lowers the ' Mary' logit by {drop:+.3f}"
)
RESULT = drop > 0

## 5. Open it in Neuronpedia's graph viewer

`to_neuronpedia` prunes the graph by influence on the logits (circuit-tracer's rule) and
writes Neuronpedia's attribution-graph JSON, which the viewer at
https://www.neuronpedia.org/graph/validator opens. The upload itself is a separate call that
needs a Neuronpedia API key; this notebook only writes the file.

In [ ]:
# recipe-test: tool=transcoders expected=True
exported = graph.to_neuronpedia(
    slug="gpt2-ioi-mary",
    model_id="gpt2-small",
    tokenizer=model.tokenizer,
    path="gpt2-ioi-mary.json",
)
print(exported["metadata"]["prompt_tokens"])
print(len(exported["nodes"]), "nodes and", len(exported["links"]), "links after pruning")
RESULT = len(exported["nodes"]) < graph.adjacency_matrix.shape[0]